# Train Model Nhận Diện Ngôn Ngữ Ký Hiệu trên Google Colab (GPU Miễn Phí)

## Hướng dẫn:
1. Upload notebook này lên Google Colab
2. Bật GPU: Runtime > Change runtime type > GPU
3. Upload dataset lên Google Drive
4. Chạy từng cell theo thứ tự

## 1. Cài đặt thư viện

In [ ]:
!pip install ultralytics -q
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Thiết lập đường dẫn dataset

Thay đổi đường dẫn phù hợp với vị trí dataset trên Drive của bạn

In [ ]:
import os

# Thay đổi đường dẫn này
DATASET_PATH = "/content/drive/MyDrive/sign_language_dataset"

# Kiểm tra dataset
if os.path.exists(DATASET_PATH):
    print(f"✓ Tìm thấy dataset tại: {DATASET_PATH}")
    
    train_path = os.path.join(DATASET_PATH, "train")
    if os.path.exists(train_path):
        classes = [d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))]
        print(f"✓ Số classes: {len(classes)}")
        print(f"✓ Classes: {classes}")
    else:
        print(f"⚠ Không tìm thấy thư mục train")
else:
    print(f"⚠ Không tìm thấy dataset. Vui lòng kiểm tra đường dẫn!")

## 4. Train model

In [ ]:
from ultralytics import YOLO

# Cấu hình
MODEL_SIZE = "yolov8n-cls.pt"  # n=nano, s=small, m=medium
EPOCHS = 100
IMAGE_SIZE = 224
BATCH_SIZE = 64  # GPU có thể dùng batch size lớn hơn

# Load model
model = YOLO(MODEL_SIZE)

# Train
results = model.train(
    data=DATASET_PATH,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,  # GPU
    patience=20,
    save=True,
    plots=True,
    verbose=True
)

print("\n✓ Training hoàn tất!")

## 5. Đánh giá model

In [ ]:
# Validate
metrics = model.val()

print(f"\nKết quả:")
print(f"  - Top-1 Accuracy: {metrics.top1:.2%}")
print(f"  - Top-5 Accuracy: {metrics.top5:.2%}")

## 6. Lưu model về Google Drive

In [ ]:
import shutil

# Copy best.pt về Drive
source = "runs/classify/train/weights/best.pt"
destination = "/content/drive/MyDrive/best.pt"

shutil.copy2(source, destination)
print(f"✓ Đã lưu model tại: {destination}")
print(f"\nTải về máy và đặt vào thư mục project để sử dụng!")

## 7. Test model trên một vài ảnh

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import numpy as np

# Load model
model = YOLO("runs/classify/train/weights/best.pt")

# Test trên validation set
val_path = os.path.join(DATASET_PATH, "val")
if os.path.exists(val_path):
    classes = os.listdir(val_path)
    
    for cls in classes[:3]:  # Test 3 classes đầu
        cls_path = os.path.join(val_path, cls)
        images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        if images:
            img_path = os.path.join(cls_path, images[0])
            img = cv2.imread(img_path)
            
            # Predict
            results = model.predict(img)[0]
            top_class = results.names[int(results.probs.top1)]
            confidence = float(results.probs.top1conf)
            
            # Hiển thị
            print(f"\nTrue: {cls} | Predicted: {top_class} ({confidence:.2%})")
            cv2_imshow(img)
else:
    print("Không tìm thấy validation set")